# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We will:
- Load the dataset's Croissant schema
- Discover record sets, fields, and columns via their `@id`s
- Extract the data into pandas DataFrames
- Conduct simple exploratory analysis
- Visualize results using matplotlib/seaborn

### Dataset Source
Croissant metadata: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dataclass, not a dict. Access attributes directly.
print(f"Dataset loaded: {getattr(metadata, 'name', '')}\n\n{getattr(metadata, 'description', '')}")

## 2. Data Overview
We explore available record sets, fields, and columns using their `@id` fields. This helps us know how to extract the data. All IDs are referenced by `@id`, not by name or index.

In [ ]:
# Get all record sets in the dataset
record_sets = dataset.record_sets
print(f"Record sets found: {len(record_sets)}\n")

for rs in record_sets:
    rs_id = rs.id
    print(f"RecordSet @id: {rs_id}\n  Name: {getattr(rs, 'name', None)}\n  Description: {getattr(rs, 'description', '')}")
    print("  Fields and their @ids:")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', None)}, DataType: {getattr(field, 'data_type', '')}")
    print("-"*60)

## 3. Data Extraction
Now load data from each record set into pandas DataFrames for analysis, referencing each by `@id`.

In [ ]:
dataframes = {}
rs_ids = [rs.id for rs in record_sets]

for rs_id in rs_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display DataFrames' column info
for rs_id, df in dataframes.items():
    print(f'RecordSet @id: {rs_id}')
    print('Columns:', df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field for analysis, filter, normalize, and group the data, referencing everything by `@id` as required.

In [ ]:
# For this demonstration, use the first available record set with numeric fields
selected_rs_id = None
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    if rs.id in dataframes:
        df = dataframes[rs.id]
        for field in getattr(rs, 'fields', []):
            # Heuristically select the first numeric field
            if getattr(field, 'data_type', '').lower() in ('number', 'integer', 'float'):
                numeric_field_id = field.id
                selected_rs_id = rs.id
                # Optionally, pick a group field as well (e.g., a categorical variable)
                for gfield in getattr(rs, 'fields', []):
                    if getattr(gfield, 'data_type', '').lower() in ('string', 'text') and gfield.id != numeric_field_id:
                        group_field_id = gfield.id
                        break
                break
    if selected_rs_id and numeric_field_id:
        break

if selected_rs_id is None or numeric_field_id is None:
    print("No suitable numeric field found.")
else:
    df = dataframes[selected_rs_id]
    print(f"Using RecordSet @id: {selected_rs_id}\nNumeric field @id: {numeric_field_id}")

    # Try to cast numeric values if needed
    if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].mean()  # Use mean as example threshold

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized_field = f"{numeric_field_id}_normalized"
    filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_field]].head())

    # Group by group_field_id if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions using matplotlib and seaborn. All fields referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if previous analytic step succeeded
if selected_rs_id and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field found, make a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- We loaded the FAIR^2 colorectal cancer clinical dataset from a Croissant schema using `mlcroissant`, referencing all entities by their `@id` fields.
- Available record sets and field structures were explored and extracted into pandas DataFrames.
- We performed numeric filtering, normalization, and group summarization, strictly referencing by `@id`.
- Simple visualizations were produced, facilitating further analysis of clinical data.

Next steps could include modelling, deep dives per clinical variable, or cross-dataset comparisons using this FAIR-compliant workflow.